# Ellipsis boundary review

Review every interior `...` as a possible speaker boundary. The notebook builds a deterministic 200-boundary sample, previews both sides using the local season folders under `downloads/`, and saves resumable labels under `data/`.

Exported transcripts are not modified.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

RUN_ID = "4bd7a1a8d9024fd08c00c0e238bcc6b8"
PAD_SECONDS = 0.25
SAMPLE_SIZE = 200
SAMPLE_SEED = 42

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

run_root = repo / "data" / "mlflow_exports" / RUN_ID
audio_root = repo / "downloads"
labels_path = repo / "data" / "ellipsis_boundary_labels.jsonl"
config = json.loads((run_root / "config.json").read_text(encoding="utf-8"))

if not (run_root / "episodes").is_dir():
    raise FileNotFoundError(f"Missing local export: {run_root}")
if not audio_root.is_dir():
    raise FileNotFoundError(f"Missing local audio: {audio_root}")

In [ ]:
from drzavni_posao.transcription.transcriber import split_ellipsis_segments


def episode_sort_key(path: Path):
    key = path.parents[1].name
    return (0, int(key)) if key.isdigit() else (1, key)


def ellipsis_boundaries(segment: dict) -> list[dict]:
    fragments = split_ellipsis_segments([segment])
    boundaries = []
    for index, (left, right) in enumerate(zip(fragments, fragments[1:])):
        split_at = left["end"]
        boundaries.append(
            {
                "ellipsis_index": index,
                "start": left["start"],
                "split_at": split_at,
                "end": right["end"],
                "local_word_gap": 2 * (right["words"][0]["start"] - split_at),
                "ellipsis_count": len(fragments) - 1,
                "multiple_ellipses": len(fragments) > 2,
                "text": segment.get("text", ""),
                "highlighted_text": f"{left['text']}⟪...⟫{right['text']}",
                "left": left["text"],
                "right": right["text"],
            }
        )
    return boundaries


def load_candidates() -> tuple[pd.DataFrame, int, int]:
    rows = []
    transcript_count = 0
    segment_count = 0
    paths = sorted(
        (run_root / "episodes").glob("*/transcription/transcript.json"),
        key=episode_sort_key,
    )
    for path in paths:
        episode_key = path.parents[1].name
        try:
            segments = json.loads(path.read_text(encoding="utf-8"))["segments"]
        except json.JSONDecodeError:
            continue

        assignment_path = path.parents[1] / "character_assigner" / "transcript.json"
        try:
            assigned = json.loads(assignment_path.read_text(encoding="utf-8"))[
                "segments"
            ]
        except (FileNotFoundError, json.JSONDecodeError):
            assigned = []

        transcript_count += 1
        segment_count += len(segments)

        for segment_index, segment in enumerate(segments):
            assigned_speaker = (
                assigned[segment_index].get("speaker")
                if segment_index < len(assigned)
                else None
            )
            for boundary in ellipsis_boundaries(segment):
                rows.append(
                    {
                        "candidate_id": (
                            f"{RUN_ID}:{episode_key}:{segment_index}:"
                            f"{boundary['ellipsis_index']}"
                        ),
                        "episode": episode_key,
                        "segment_index": segment_index,
                        "start": segment["start"],
                        "end": segment["end"],
                        "assigned_speaker": assigned_speaker,
                        **boundary,
                    }
                )

    return pd.DataFrame(rows), transcript_count, segment_count


probe = ellipsis_boundaries(
    {
        "start": 0.0,
        "end": 3.0,
        "text": "A... B... C",
        "words": [
            {"word": "A...", "start": 0.0, "end": 0.5},
            {"word": "B...", "start": 1.0, "end": 1.5},
            {"word": "C", "start": 2.0, "end": 2.5},
        ],
    }
)
assert [(boundary["left"], boundary["right"]) for boundary in probe] == [
    ("A", "B"),
    ("B", "C"),
]

candidates, transcript_count, segment_count = load_candidates()
segment_candidates = candidates.drop_duplicates(["episode", "segment_index"])
display(
    pd.Series(
        {
            "transcripts scanned": transcript_count,
            "segments scanned": segment_count,
            "segments with word-gap boundary": len(segment_candidates),
            "share of segments": len(segment_candidates) / segment_count
            if segment_count
            else 0,
            "word-gap boundaries": len(candidates),
            "segments with multiple ellipses": int(
                segment_candidates["multiple_ellipses"].sum()
            ),
        },
        name="value",
    ).to_frame()
)

In [ ]:
def build_review_queue(
    candidates: pd.DataFrame,
    size: int = SAMPLE_SIZE,
    seed: int = SAMPLE_SEED,
) -> pd.DataFrame:
    minimums = [
        group.sample(min(10, len(group)), random_state=seed + offset)
        for offset, (_, group) in enumerate(
            candidates.groupby("multiple_ellipses", sort=True)
        )
    ]
    selected = pd.concat(minimums) if minimums else candidates.iloc[:0]
    slots = max(0, min(size, len(candidates)) - len(selected))
    remaining = candidates[~candidates["candidate_id"].isin(selected["candidate_id"])]
    if slots:
        selected = pd.concat(
            [selected, remaining.sample(slots, random_state=seed)],
            ignore_index=True,
        )
    return selected.sample(frac=1, random_state=seed).reset_index(drop=True)


def load_labels() -> dict[str, dict]:
    if not labels_path.exists():
        return {}
    records = [
        json.loads(line)
        for line in labels_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    return {record["candidate_id"]: record for record in records}


def save_labels(records: dict[str, dict]) -> None:
    labels_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = labels_path.with_suffix(".tmp")
    payload = "\n".join(
        json.dumps(record, ensure_ascii=False, sort_keys=True)
        for record in records.values()
    )
    temporary_path.write_text(payload + ("\n" if payload else ""), encoding="utf-8")
    temporary_path.replace(labels_path)


review = build_review_queue(candidates)
labels = load_labels()
assert review["candidate_id"].is_unique
assert len(review) == min(SAMPLE_SIZE, len(candidates))

pd.set_option("display.max_colwidth", 160)
print(f"Review queue: {len(review)} boundaries; saved labels: {len(labels)}")
display(
    review[
        [
            "episode",
            "segment_index",
            "ellipsis_index",
            "start",
            "split_at",
            "end",
            "local_word_gap",
            "assigned_speaker",
            "ellipsis_count",
            "highlighted_text",
        ]
    ]
)

In [ ]:
import ipywidgets as widgets
from IPython.display import Audio, clear_output

from drzavni_posao.transcription import MediaStore
from drzavni_posao.transcription.constants import SR

media = MediaStore(
    audio_root,
    run_root,
    trim_start=config["audio"]["trim_start"],
    trim_end=config["audio"]["trim_end"],
    catalog_path=repo / "data" / "youtube_episodes.json",
)

position = next(
    (
        index
        for index, candidate_id in enumerate(review["candidate_id"])
        if candidate_id not in labels
    ),
    0,
)
output = widgets.Output()
status = widgets.HTML()
speaker_choice = widgets.ToggleButtons(
    options=[
        ("Two speakers", "two_speakers"),
        ("Single speaker", "single_speaker"),
        ("Unclear / skip", "unclear"),
    ],
    value=None,
)
alignment_choice = widgets.ToggleButtons(
    options=[
        ("Alignment OK", "none"),
        ("Boundary time wrong", "boundary_timestamp_wrong"),
        ("Transcript misaligned", "transcript_alignment_wrong"),
    ],
    value="none",
)
previous_button = widgets.Button(description="Previous")
replay_button = widgets.Button(description="Replay")
next_button = widgets.Button(description="Next")
save_button = widgets.Button(description="Save & Next", button_style="success")
buttons = [previous_button, replay_button, next_button, save_button]


def show_candidate():
    with output:
        clear_output(wait=True)
        if review.empty:
            print("No matching boundaries.")
            for button in buttons:
                button.disabled = True
            return

        row = review.iloc[position]
        saved = labels.get(row.candidate_id)
        speaker_choice.value = saved["speaker_outcome"] if saved else None
        alignment_choice.value = saved["alignment_issue"] if saved else "none"
        reviewed_count = sum(
            candidate_id in labels for candidate_id in review["candidate_id"]
        )
        print(
            f"{position + 1}/{len(review)}  reviewed {reviewed_count}/{len(review)}  "
            f"episode {row.episode}  segment {row.segment_index}  "
            f"ellipsis {row.ellipsis_index + 1}/{row.ellipsis_count}"
        )
        print(
            f"{row.start:.3f}–{row.end:.3f}s  split={row.split_at:.3f}s  "
            f"gap={row.local_word_gap:.3f}s"
        )
        print(f"current assignment: {row.assigned_speaker or '[none]'}")
        print(row.highlighted_text)
        print(f"LEFT:  {row.left}")
        print(f"RIGHT: {row.right}")

        try:
            episode = media.episode(str(row.episode))
        except KeyError as error:
            print(error)
            return

        waveform = episode.audio()
        clip_start = max(0, int((row.start - PAD_SECONDS) * SR))
        split_at = int(row.split_at * SR)
        clip_end = min(len(waveform), int((row.end + PAD_SECONDS) * SR))
        full_clip = waveform[clip_start:clip_end].copy()
        left_clip = waveform[clip_start : min(clip_end, split_at)].copy()
        right_clip = waveform[max(clip_start, split_at) : clip_end].copy()
        episode.clear_audio()

        print("FULL")
        display(Audio(full_clip, rate=SR, autoplay=True))
        print("PROPOSED LEFT / RIGHT")
        display(Audio(left_clip, rate=SR), Audio(right_clip, rate=SR))

    status.value = "<b>Saved</b>" if saved else "Not reviewed"


def step(delta: int):
    global position
    position = max(0, min(len(review) - 1, position + delta))
    show_candidate()


def save_current(_):
    global position
    if speaker_choice.value is None:
        status.value = "<b style='color:#b00'>Choose a speaker outcome.</b>"
        return

    row = review.iloc[position]
    labels[row.candidate_id] = {
        "candidate_id": row.candidate_id,
        "run_id": RUN_ID,
        "episode": str(row.episode),
        "segment_index": int(row.segment_index),
        "ellipsis_index": int(row.ellipsis_index),
        "start": float(row.start),
        "split_at": float(row.split_at),
        "end": float(row.end),
        "local_word_gap": float(row.local_word_gap),
        "multiple_ellipses": bool(row.multiple_ellipses),
        "text": row.text,
        "speaker_outcome": speaker_choice.value,
        "alignment_issue": alignment_choice.value,
    }
    save_labels(labels)

    unreviewed = [
        index
        for index, candidate_id in enumerate(review["candidate_id"])
        if candidate_id not in labels
    ]
    if unreviewed:
        position = next(
            (index for index in unreviewed if index > position),
            unreviewed[0],
        )
    show_candidate()


previous_button.on_click(lambda _: step(-1))
replay_button.on_click(lambda _: show_candidate())
next_button.on_click(lambda _: step(1))
save_button.on_click(save_current)
display(
    widgets.VBox(
        [
            output,
            widgets.HTML("<b>Speaker outcome</b>"),
            speaker_choice,
            widgets.HTML("<b>Alignment</b>"),
            alignment_choice,
            status,
            widgets.HBox(buttons),
        ]
    )
)
show_candidate()

In [ ]:
# Rerun this cell at any point to summarize saved labels.
reviewed = pd.DataFrame(
    [
        labels[candidate_id]
        for candidate_id in review["candidate_id"]
        if candidate_id in labels
    ]
)
if reviewed.empty:
    print(f"0/{len(review)} boundaries reviewed")
else:
    clear = reviewed[reviewed["speaker_outcome"] != "unclear"].copy()
    two_speaker_count = int((reviewed["speaker_outcome"] == "two_speakers").sum())
    display(
        pd.Series(
            {
                "reviewed": len(reviewed),
                "remaining": len(review) - len(reviewed),
                "clear speaker outcome": len(clear),
                "unclear / skipped": int(
                    (reviewed["speaker_outcome"] == "unclear").sum()
                ),
                "two speakers": two_speaker_count,
                "single speaker": int(
                    (reviewed["speaker_outcome"] == "single_speaker").sum()
                ),
                "two-speaker share among clear": (
                    two_speaker_count / len(clear) if len(clear) else None
                ),
            },
            name="value",
        ).to_frame()
    )
    display(reviewed["alignment_issue"].value_counts().rename("count").to_frame())
    if not clear.empty:
        clear["is_two_speakers"] = clear["speaker_outcome"] == "two_speakers"
        display(
            clear.groupby("multiple_ellipses").agg(
                boundaries=("candidate_id", "size"),
                two_speaker_rate=("is_two_speakers", "mean"),
            )
        )

        with_gaps = clear[clear["local_word_gap"].notna()].copy()
        if not with_gaps.empty:
            with_gaps["gap_bucket"] = pd.cut(
                with_gaps["local_word_gap"],
                [-float("inf"), 0, 0.1, 0.3, 1.0, float("inf")],
            )
            display(
                with_gaps.groupby("gap_bucket", observed=True).agg(
                    boundaries=("candidate_id", "size"),
                    two_speaker_rate=("is_two_speakers", "mean"),
                )
            )

In [ ]:
# Review segments with both weak ASR probability and forced-alignment confidence.
LOW_AVG_LOGPROB = -0.5
LOW_MEAN_WORD_SCORE = 0.5
REQUIRE_BOTH = True


def load_low_confidence_segments() -> pd.DataFrame:
    rows = []
    paths = sorted(
        (run_root / "episodes").glob("*/transcription/transcript.json"),
        key=episode_sort_key,
    )
    for path in paths:
        episode_key = path.parents[1].name
        try:
            segments = json.loads(path.read_text(encoding="utf-8"))["segments"]
        except json.JSONDecodeError:
            continue

        assignment_path = path.parents[1] / "character_assigner" / "transcript.json"
        try:
            assigned = json.loads(assignment_path.read_text(encoding="utf-8"))[
                "segments"
            ]
        except (FileNotFoundError, json.JSONDecodeError):
            assigned = []

        for index, segment in enumerate(segments):
            avg_logprob = segment.get("avg_logprob")
            word_scores = [
                word["score"]
                for word in segment.get("words", [])
                if word.get("score") is not None
            ]
            mean_word_score = (
                sum(word_scores) / len(word_scores) if word_scores else None
            )
            low_probability = avg_logprob is not None and avg_logprob <= LOW_AVG_LOGPROB
            low_alignment = (
                mean_word_score is not None and mean_word_score <= LOW_MEAN_WORD_SCORE
            )
            selected = (
                low_probability and low_alignment
                if REQUIRE_BOTH
                else low_probability or low_alignment
            )
            if not selected:
                continue

            reasons = []
            if low_probability:
                reasons.append("ASR probability")
            if low_alignment:
                reasons.append("alignment confidence")
            rows.append(
                {
                    "episode": episode_key,
                    "segment_index": index,
                    "start": segment["start"],
                    "end": segment["end"],
                    "duration": segment["end"] - segment["start"],
                    "avg_logprob": avg_logprob,
                    "mean_word_score": mean_word_score,
                    "reason": " + ".join(reasons),
                    "assigned_speaker": (
                        assigned[index].get("speaker")
                        if index < len(assigned)
                        else None
                    ),
                    "text": segment.get("text", ""),
                }
            )

    if not rows:
        return pd.DataFrame(rows)
    return (
        pd.DataFrame(rows)
        .sort_values(
            ["avg_logprob", "mean_word_score"],
            na_position="last",
        )
        .reset_index(drop=True)
    )


low_confidence = load_low_confidence_segments()
operator = "and" if REQUIRE_BOTH else "or"
print(
    f"{len(low_confidence)} segments matched "
    f"(avg_logprob <= {LOW_AVG_LOGPROB} {operator} "
    f"mean word score <= {LOW_MEAN_WORD_SCORE})"
)
display(low_confidence.head(200))

low_position = 0
low_output = widgets.Output()
low_previous = widgets.Button(description="Previous")
low_replay = widgets.Button(description="Replay")
low_next = widgets.Button(description="Next")
low_buttons = [low_previous, low_replay, low_next]


def show_low_confidence():
    with low_output:
        clear_output(wait=True)
        if low_confidence.empty:
            print("No low-confidence segments at these thresholds.")
            for button in low_buttons:
                button.disabled = True
            return

        row = low_confidence.iloc[low_position]
        print(
            f"{low_position + 1}/{len(low_confidence)}  episode {row.episode}  "
            f"segment {row.segment_index}  {row.start:.3f}–{row.end:.3f}s"
        )
        print(
            f"avg_logprob={row.avg_logprob:.3f}  "
            f"mean_word_score={row.mean_word_score:.3f}  reason={row.reason}"
        )
        print(f"current assignment: {row.assigned_speaker or '[none]'}")
        print(row.text)

        try:
            episode = media.episode(str(row.episode))
        except KeyError as error:
            print(error)
            return

        waveform = episode.audio()
        clip_start = max(0, int((row.start - PAD_SECONDS) * SR))
        clip_end = min(len(waveform), int((row.end + PAD_SECONDS) * SR))
        clip = waveform[clip_start:clip_end].copy()
        episode.clear_audio()
        display(Audio(clip, rate=SR, autoplay=True))


def step_low_confidence(delta: int):
    global low_position
    low_position = max(
        0,
        min(len(low_confidence) - 1, low_position + delta),
    )
    show_low_confidence()


low_previous.on_click(lambda _: step_low_confidence(-1))
low_replay.on_click(lambda _: show_low_confidence())
low_next.on_click(lambda _: step_low_confidence(1))
display(widgets.VBox([low_output, widgets.HBox(low_buttons)]))
show_low_confidence()